# Project Report — Fumo & Chocopuni Image Classifier

# Introduction and description of the dataset


This project tackles a niche but visually interesting problem: distinguishing between
two types of plush toys — **Fumo** and **Chocopuni**. They differ
significantly in style, shape, and manufacturer.

| | Fumo (FumoFumo) | Chocopuni (Chokopuni) |
|---|---|---|
| **Manufacturer** | Gift (official) | Good Smile Company |
| **Style** | Classic, slightly angular, "mofumofu" | Round, chibi, soft "puni" shape |
| **Size** | ~20–25 cm (seated) | Smaller, more compact |
| **Face** | Flat face, embroidered eyes | Softer features, printed eyes |

The two types have distinct silhouettes,
proportions, and textures — making this a realistic fine-grained visual classification task.


## Dataset

The Fumo and Chocopuni datasets consist of approximately 90% pictures taken by myself and friends.
The remaining images were scraped from the internet and used primarily as test data —
the goal being to verify that the model can recognize plushies it has never seen before,
not just the ones it was trained on. This gives a much more honest measure of real-world performance.

📁 **Dataset:** [Link to dataset](https://drive.google.com/drive/folders/1gp_d7FjI8-9Uuu579UwrCWxPdw3DeMX8)




### Dataset Collection

With only 4 chocopunis and 11 fumos available in my inventory, reaching a meaningful dataset size
without causing overfitting was a real constraint.

Collection happened in three stages:

1. **Controlled environment** — black background, white light, rotating the plushie to capture all angles
2. **Second controlled environment** — white background, warm light, still clean studio-style shots
3. **Random environments** — various rooms, lighting conditions, backgrounds, and positions to simulate real-world photos

This brought the count to roughly ~80 images per class. To go further, I reached out to
friends and people I know who also own these type of plushies, which added more unique subjects
and backgrounds to the dataset.

## Model Performance Summary

| | Model 1 — Custom CNN | Model 2 — EfficientNet-B0 | Model 3 — Fine-Tuned |
|---|---|---|---|
| **Test Accuracy** | 61.0% | 87.5% | 91.7% |
| **Loss** | 0.671 | 0.355 | 0.392 |
| **Macro F1** | 0.60 | 0.88 | 0.92 |
| **Chocopuni F1** | 0.65 | 0.88 | 0.92 |
| **Fumo F1** | 0.56 | 0.88 | 0.91 |
| **Bias** | Strong toward Chocopuni | Balanced | Balanced |

## Model 2 — EfficientNet-B0 as Feature Extractor

Switching to a pretrained EfficientNet-B0 backbone produced a dramatic improvement.
Test accuracy jumped to **87.5%** — a gain of **+26.5 percentage points** over Model 1.
EfficientNet-B0 was selected for its balance of speed and efficiency.

Key differences from Model 1:
- The fully frozen EfficientNet backbone provided rich, generalizable feature representations
  (edges, textures, shapes) learned from 1.2 million ImageNet images
- Only the classifier head was trained on our data — a small fraction of the total 4 million parameters
- Stratified 5-fold cross-validation replaced the fixed train/val split,
  giving a more reliable estimate of generalization on a small dataset

The class bias present in Model 1 largely disappeared: both classes reached F1
scores of 0.88, and the model showed no meaningful preference for either class.

## Model 3 — Fine-Tuned EfficientNet-B0

Fine-tuning the upper three convolutional blocks (features[6], [7], [8]) pushed
accuracy further to **91.7%** — a gain of **+6.3 percentage points** over Model 2.

Both classes are now nearly perfectly balanced (F1: 0.92 and 0.91), and the loss
dropped to 0.392, indicating higher prediction confidence. The diminishing returns
between Model 2 and 3 suggest the dataset is approaching its practical performance
ceiling — further gains would likely require more training data.


## Hyperparameter Exploration

Models 2 and 3 were tested under several different configurations during development:

| Parameter | Values tested | Model 2 final | Model 3 final |
|---|---|---|---|
| `BATCH_SIZE` | 16, 32, 64 | 32 | 32 |
| `EPOCHS` | 20, 30, 40 | 30 | 40 |
| `NUM_WORKERS` | 4, 8, 10 | 10 | 4 |
| `N_FOLDS` | 3, 5 | 5 | 5 |
| `LR` | 1e-3, 1e-4 | 1e-3 | 1e-3 (head), 1e-6 (conv) |

Key observations from this exploration:
- **5 folds** was used for both models — more folds give more reliable estimates on a small dataset
- **30 epochs** for Model 2 was sufficient; the fully frozen backbone converges quickly since
  only a small classifier head is being trained
- **40 epochs** were needed for Model 3 because fine-tuning pretrained layers with a very
  low learning rate (`1e-6`) converges significantly slower
- **NUM_WORKERS > 4** on Windows caused `WinError 1455` (paging file too small) due to each
  worker spawning a full PyTorch process; this was resolved differently on the two machines
  used during development

---

## Challenges

### Switching to PyTorch

The project started with Keras (PyTorch backend) but encountered kernel crashes
related to the `Permute` layer and multiprocessing conflicts on Windows. From
Model 2 onwards the project moved to pure PyTorch with a manual training loop.
This required learning gradient handling, DataLoader mechanics, and writing
evaluation loops from scratch — but resulted in a more transparent and
debuggable setup.

### Dataset collection

The physical collection was limited by the number of plushies available (4 Chocopuni, 11 Fumo).
Shooting in multiple environments (controlled studio vs. random real-world) was intentional
but introduced a train/test distribution mismatch: training images were mostly
clean studio shots while test images included real-world backgrounds. This likely
contributed to Model 1's poor generalization and was only partially resolved by
adding varied-background images in later dataset iterations.

### Training Speed & GPU Utilization

One of the more unexpected challenges was training speed. Despite having powerful GPU, training Model 3 initially took over 35 minutes for 5 folds × 40 epochs —
far longer than expected.

Investigation revealed that **GPU utilization was stuck at ~12%**. The bottleneck was
not compute but the CPU→GPU data transfer: with only ~280 images, each batch was
processed by the GPU in microseconds, then the GPU sat idle waiting for the next
batch to be loaded from disk, augmented on the CPU, and transferred.

The fix was to preload the entire dataset directly into VRAM at startup:

```python
def preload_to_gpu(dataset, device):
    loader = DataLoader(dataset, batch_size=len(dataset), num_workers=0)
    images, labels = next(iter(loader))
    return TensorDataset(images.to(device), labels.to(device))
```

This eliminated all CPU→GPU transfers during training. Training time dropped from
**35 minutes to ~30 seconds** — roughly a 70× speedup.

However, this introduced a subtle problem: **augmentation was now frozen**. When the
dataset is preloaded, augmentation is applied once at load time and then the same
transformed images are reused every epoch. This breaks the regularization effect of
augmentation, which normally shows each image in a slightly different form each epoch.

For Model 2 this had minimal impact — with only 2,562 trainable parameters (just the
classifier head), there is almost nothing to overfit, and the frozen backbone does the
heavy lifting regardless of augmentation variety. Model 2 still achieved 85.4% test accuracy.

For Model 3, the frozen augmentation was more significant because more parameters were
unfrozen and could potentially overfit. A trial run with GPU preloading enabled showed
Model 3 dropping from **93% to ~85%** — confirming that augmentation diversity matters
more when fine-tuning. After recognizing this, the GPU preloading approach was rolled back
for Model 3 and the standard DataLoader with CPU augmentation was restored. The
30-second training time was lost, but accuracy recovered to **91.7%**.

The final takeaway: GPU preloading is a valid optimization for pure feature extraction
(Model 2), but should be used with caution when augmentation is the primary regularizer.

### Windows multiprocessing and memory

Running DataLoader with `NUM_WORKERS > 4` on Windows caused `WinError 1455`
(paging file too small) because each worker spawns a full PyTorch process.
Reducing `NUM_WORKERS` to 4 resolved the issue with no meaningful impact on
training speed given the dataset size.

## Model 1

Training accuracy in Model 1 exceeded 98% within the first few epochs while
validation accuracy stagnated — a textbook case of overfitting on a small dataset.
Dropout (0.5), data augmentation, and early stopping were applied, but without
pretrained features the model still failed to learn meaningful representations.


![Overfitting](images/overfitting.png)


The confusion matrix revealed the real problem: the model had a strong bias toward
predicting chocopuni, missing roughly half of all fumo images. This pointed to both
class imbalance and a mismatch between the training and test distributions -
training images were shot on controlled backgrounds, while test images came from
varied real-world settings.







![picture](images/m1.png)

## Model 2

Compared to Model 1, the confusion matrix shows a dramatically more balanced
prediction pattern. The model correctly identifies the majority of both classes,
with only a small number of errors in each direction.

- **4 Fumo images** were incorrectly predicted as Chocopuni (recall 0.95)
- **5 Chocopuni images** were incorrectly predicted as Fumo (recall 0.81)
- The near-diagonal matrix confirms the model is no longer biased toward
  either class — errors are scattered rather than systematic

The remaining errors likely correspond to ambiguous photos: unusual angles,
heavy occlusion, or lighting conditions not well represented in training data.

![picture](images/cm2.png)

### Training Results — K-Fold Cross-Validation

The model was trained using 5-fold stratified cross-validation, 30 epochs per fold.
No early stopping triggered — all folds ran the full 30 epochs with steadily
decreasing validation loss.

| Fold | Val Accuracy | Val Loss (best) |
|------|-------------|-----------------|
| Fold 1 | 100.0% | 0.0804 |
| Fold 2 | 96.4% | 0.1708 |
| Fold 3 | 92.9% | 0.2023 |
| Fold 4 | 98.2% | 0.1502 |
| Fold 5 | 96.4% | 0.1708 |
| **Mean** | **96.8%** | — |
| **Std dev** | **±2.4%** | — |

The low standard deviation (±2.4%) indicates stable performance across different
data splits. **Fold 1** produced the best checkpoint and was selected for final evaluation.

### Test Results

The best fold checkpoint was evaluated on the held-out test set,
which was never seen during training or fold selection:

| Metric | Value |
|--------|-------|
| **Test Accuracy** | 87.5% |
| **Test Loss** | 0.355 |

### Per-Class Results

| Class | Precision | Recall | F1 | Support |
|-------|-----------|--------|----|---------|
| Chocopuni | 0.95 | 0.81 | 0.88 | 26 |
| Fumo | 0.81 | 0.95 | 0.88 | 22 |
| **Accuracy** | | | **0.88** | **48** |
| Macro avg | 0.88 | 0.88 | 0.88 | 48 |

The gap between mean validation accuracy (96.8%) and test accuracy (87.5%)
is about 9 percentage points, expected given that the test set contains
real-world images from the internet with much more varied backgrounds and
angles than the training photos.

![picture](images/m2.png)

## Model 3

### Training Results — K-Fold Cross-Validation

The model was trained using 5-fold stratified cross-validation with up to 40 epochs
per fold. Unlike Model 2, no early stopping triggered — all folds ran the full 40
epochs, reflecting the slower convergence expected when fine-tuning pretrained layers
with a very low learning rate (1e-6).

| Fold | Val Accuracy | Val Loss (best) |
|------|-------------|-----------------|
| Fold 1 | 100.0% | 0.1532 |
| Fold 2 | 92.9% | 0.2734 |
| Fold 3 | 92.9% | 0.2704 |
| Fold 4 | 94.6% | 0.2584 |
| Fold 5 | 91.1% | 0.2691 |
| **Mean** | **94.3%** | — |
| **Std dev** | **±3.1%** | — |

**Fold 1** achieved 100% validation accuracy and was selected as the best checkpoint.
This result should be interpreted with caution — the validation fold of 57 images is
small enough that a perfect score may reflect a particularly easy split rather than
true generalization. The higher standard deviation (±3.1%) compared to Model 2 (±1.8%)
confirms greater sensitivity to which images end up in each fold, which is expected
when more layers are being fine-tuned.

### Test Results

The best fold checkpoint (Fold 1) was evaluated on the held-out test set:

| Metric | Value |
|--------|-------|
| **Test Accuracy** | 91.7% |
| **Test Loss** | 0.392 |

The gap between mean validation accuracy (94.3%) and test accuracy (91.7%) is
approximately 2.6 percentage points — notably smaller than the 7.8 pp gap in
Model 2. This suggests that fine-tuning produced representations that generalize
better to real-world images from the internet, not just held-out studio photos.




![picture](images/m3.png)

### Per-Class Results

| Class | Precision | Recall | F1 | Support |
|-------|-----------|--------|----|---------|
| Chocopuni | 0.96 | 0.88 | 0.92 | 26 |
| Fumo | 0.88 | 0.95 | 0.91 | 22 |
| **Accuracy** | | | **0.92** | **48** |
| Macro avg | 0.92 | 0.92 | 0.92 | 48 |

Both classes are nearly perfectly balanced. Chocopuni precision (0.96) is the
highest single metric across all three models — when the model predicts Chocopuni,
it is almost always correct.

![picture](images/cm3.png)


## Reflection

These results show that transfer learning works best for small image datasets.
Model 1 (trained from scratch) quickly overfits and cannot reliably distinguish between Fumo and Chocopuni with only ~300 images.

This task is difficult because it is a fine-grained classification problem.
Both classes look very similar — same characters, colors, and style.
The differences are small (shape of the head, face, proportions, legs), so the model needs to understand shapes, not just colors or textures.

That’s why pretrained models help so much.
Models trained on ImageNet already know how to detect shapes and objects, so they perform better.

Model 2 and Model 3 both use pretrained features.
The difference is that Model 3 allows more layers to adapt.
The improvement (+6.3%) suggests that the model learned details specific to plush toys.

In practice, the best approach is:

- start with a pretrained model (Model 2)
- then fine-tune it (Model 3) if needed

To improve results further, the most important thing is more and better data, not a more complex model.

![picture](images/mc.png)